# Lab 2 · Evaluate RAG (free Kaggle T4)

**Layer 3 · Data.** "It looked right" is not evidence. RAG has **two** stages that
can each fail, so we measure both on the same **Kubernetes / SRE runbook** corpus
from Lab 1:

| Stage | Question | Metric here |
|-------|----------|-------------|
| **Retrieval** | did we fetch the right runbook? | recall@k, MRR |
| **Generation** | is the answer faithful to it? | faithfulness, answer-relevance (LLM-as-judge) |

Then we **sweep the knobs** — top-k, a reranker, chunk size — and read the
trade-offs, exactly the decisions the README's Phase 7 describes.

**Settings:** Accelerator = **GPU (T4)**, Internet = **On**. Then **Run All**.

**Chaining:** *Add Input → Notebook Output → your Lab 1 version* to reuse its
index + chunks + Q&A. If nothing is attached, cell 1 **rebuilds inline**, so this
lab is self-contained.

In [ ]:
!pip install -q faiss-cpu sentence-transformers

import torch, faiss, json, os, glob, textwrap
import numpy as np
import pandas as pd
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('cuda', torch.cuda.is_available())

## 1 · Load Lab 1 outputs — or rebuild inline

We look for `qa.jsonl` under `/kaggle/input/` (Lab 1's saved output). If absent, we
rebuild the identical Globex SRE corpus so the lab still runs standalone. Either
way we end up with `DOCS`, `QA`, and the build config.

In [ ]:
DOCS = [
  {'id': 'k8s-pod-failures', 'title': 'Runbook: pod failures',
   'text': ('A pod in CrashLoopBackOff is starting and crashing repeatedly; the kubelet backs off restarts up to 5 minutes. '
     'An OOMKilled pod was terminated for exceeding its memory limit and shows exit code 137. '
     'ImagePullBackOff means the image cannot be pulled, usually a bad tag or missing registry credentials. '
     'Globex runbook: for OOMKilled in the payments namespace, raise the memory limit to 2Gi, redeploy, and page the #sre-payments channel.')},
  {'id': 'k8s-probes', 'title': 'Runbook: health probes',
   'text': ('A liveness probe restarts a container that is alive but stuck; a readiness probe pulls a pod out of the Service endpoints until it can serve traffic. '
     'A startup probe protects slow-starting containers so the liveness probe does not kill them too early. '
     'Globex default: liveness is an httpGet on /healthz every 10 seconds with a failure threshold of 3; readiness is an httpGet on /ready.')},
  {'id': 'k8s-scaling', 'title': 'Runbook: autoscaling and resources',
   'text': ('Requests are what the scheduler reserves; limits are the hard ceiling the kubelet enforces, and exceeding the memory limit triggers an OOMKill. '
     'The HorizontalPodAutoscaler scales replicas on CPU or custom metrics. '
     'The Globex web tier scales between 3 and 30 replicas at a target of 70 percent CPU. '
     'The cluster autoscaler adds nodes from the burst node pool when pods stay unschedulable for more than 2 minutes.')},
  {'id': 'k8s-oncall', 'title': 'Runbook: on-call and severity',
   'text': ('Globex uses three severity levels: SEV1 is a full outage, SEV2 is degraded service, SEV3 is a minor issue. '
     'A SEV1 pages the primary on-call immediately and the secondary after 10 minutes. '
     'The automated alert threshold is 3 pod restarts within 10 minutes. '
     'On-call rotations are weekly and hand over every Monday at 10:00.')},
  {'id': 'k8s-deploy', 'title': 'Runbook: deployments and rollback',
   'text': ('Globex deploys with a rolling update: max surge 25 percent, max unavailable 0. '
     'Every production deploy must pass a canary that takes 5 percent of traffic for 15 minutes before full rollout. '
     'To roll back, run kubectl rollout undo on the deployment, which reverts to the previous ReplicaSet. '
     'Deploys are frozen on Fridays after 14:00 and during any SEV1 or SEV2 incident.')},
  {'id': 'incident-2026-04', 'title': 'Postmortem: payments outage 12 Apr 2026',
   'text': ('On 12 April 2026 the payments-api entered CrashLoopBackOff after release v2.3.1 introduced a memory leak. '
     'Pods were OOMKilled with exit code 137 about every 4 minutes, tripping the restart alert. '
     'The on-call raised the memory limit to 2Gi as a stopgap and rolled back to v2.3.0 with kubectl rollout undo. '
     'Root cause: an unbounded in-memory cache; the fix shipped in v2.3.2 with a cache size cap.')},
]

def find_input(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

qa_path = find_input('qa.jsonl')
if qa_path:
    QA = [json.loads(l) for l in open(qa_path)]
    print(f'loaded {len(QA)} Q&A pairs from Lab 1 output')
else:
    QA = [
      {'question': 'For an OOMKilled pod in the payments namespace, what memory limit should I set?', 'gold_doc': 'k8s-pod-failures'},
      {'question': 'What exit code does an OOMKilled pod show?', 'gold_doc': 'k8s-pod-failures'},
      {'question': 'What is the automated alert threshold for pod restarts at Globex?', 'gold_doc': 'k8s-oncall'},
      {'question': 'How do I roll back a bad Kubernetes deployment at Globex?', 'gold_doc': 'k8s-deploy'},
      {'question': 'How much traffic does a Globex canary take, and for how long?', 'gold_doc': 'k8s-deploy'},
      {'question': 'What caused the payments outage on 12 April 2026?', 'gold_doc': 'incident-2026-04'},
      {'question': 'What are the three Globex severity levels?', 'gold_doc': 'k8s-oncall'},
      {'question': 'What is the Globex default liveness probe configuration?', 'gold_doc': 'k8s-probes'},
    ]
    print(f'no Lab 1 input attached - rebuilding inline ({len(QA)} Q&A pairs)')

## 2 · Build helpers (embed · chunk · index · retrieve)

We parametrize the build by **chunk size** so the sweep can rebuild the index
cheaply. Same `bge-small` embedder and same query prefix as Lab 1 — change either
and the numbers below would shift.

In [ ]:
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
EMB_ID = 'BAAI/bge-small-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
tok = AutoTokenizer.from_pretrained(MODEL_ID)
embedder = SentenceTransformer(EMB_ID, device=DEVICE)

def chunk_text(text, size, overlap):
    ids = tok.encode(text, add_special_tokens=False); step = size - overlap; out = []
    for s in range(0, len(ids), step):
        piece = ids[s:s+size]
        if not piece: break
        out.append(tok.decode(piece))
        if s + size >= len(ids): break
    return out

def build(chunk_size=64, overlap=12):
    chunks = []
    for d in DOCS:
        for j, c in enumerate(chunk_text(d['text'], chunk_size, overlap)):
            chunks.append({'chunk_id': f"{d['id']}::{j}", 'doc_id': d['id'], 'text': c})
    vecs = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                           convert_to_numpy=True).astype('float32')
    idx = faiss.IndexFlatIP(vecs.shape[1]); idx.add(vecs)
    return chunks, idx

def retrieve(query, chunks, idx, k=5):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = idx.search(qv, k)
    return [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]

chunks, index = build(64, 12)
print('built', len(chunks), 'chunks')

## 3 · Retrieval metrics (does the right runbook come back?)

For each question we know the **gold document**. Two standard metrics:

- **recall@k** — fraction of questions whose gold doc appears in the top-k.
- **MRR** — mean of `1/rank` of the first gold hit (1.0 = always rank 1). Rewards
  putting the right chunk *first*, not just somewhere in the list.

In [ ]:
def retrieval_metrics(chunks, idx, k):
    recall, rr = 0, 0.0
    for q in QA:
        hits = retrieve(q['question'], chunks, idx, k=k)
        docs = [h['doc_id'] for h in hits]
        if q['gold_doc'] in docs:
            recall += 1
            rr += 1.0 / (docs.index(q['gold_doc']) + 1)
    n = len(QA)
    return {'k': k, 'recall@k': round(recall/n, 3), 'MRR': round(rr/n, 3)}

pd.DataFrame([retrieval_metrics(chunks, index, k) for k in (1, 2, 3, 5)])

**Read it:** recall climbs with k (more chances to include the gold runbook) but
MRR barely moves once the right chunk is already near the top. Bigger k isn't free
— it adds noise and tokens to the prompt. This is the top-k trade-off from Phase 5.

## 4 · Add a reranker (Phase 5 · the precision lever)

Vector search is cheap but blunt. A **cross-encoder reranker** reads the query and
each chunk *together* and rescores them. We over-fetch (k=5), rerank, and keep the
top-2 — and watch MRR jump.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('BAAI/bge-reranker-base', device=DEVICE)

def rerank_metrics(chunks, idx, fetch=5, keep=2):
    recall, rr = 0, 0.0
    for q in QA:
        hits = retrieve(q['question'], chunks, idx, k=fetch)
        pairs = [[q['question'], h['text']] for h in hits]
        for h, s in zip(hits, reranker.predict(pairs)):
            h['rr_score'] = float(s)
        hits = sorted(hits, key=lambda h: h['rr_score'], reverse=True)[:keep]
        docs = [h['doc_id'] for h in hits]
        if q['gold_doc'] in docs:
            recall += 1; rr += 1.0 / (docs.index(q['gold_doc']) + 1)
    n = len(QA)
    return {'recall@2': round(recall/n, 3), 'MRR': round(rr/n, 3)}

baseline = retrieval_metrics(chunks, index, k=2)
print('vector top-2     :', {'recall@k': baseline['recall@k'], 'MRR': baseline['MRR']})
print('rerank 5->keep 2 :', rerank_metrics(chunks, index, fetch=5, keep=2))

## 5 · Chunk-size sweep (Phase 2 · the highest-leverage knob)

Rebuild the index at several chunk sizes and re-measure retrieval. Tiny chunks are
precise but fragment facts; large chunks keep context but blur what matches.

In [ ]:
rows = []
for size in (32, 64, 128, 256):
    ov = max(4, size // 6)
    ch, ix = build(size, ov)
    m = retrieval_metrics(ch, ix, k=3)
    rows.append({'chunk_size': size, 'overlap': ov, 'n_chunks': len(ch),
                 'recall@3': m['recall@k'], 'MRR': m['MRR']})
pd.DataFrame(rows)

## 6 · Generation eval — faithfulness & relevance (LLM-as-judge)

Retrieval can be perfect and the *answer* still wrong. We generate RAG answers,
then use **Qwen itself as a small judge** to score each on two axes:

- **Faithfulness** — is every claim supported by the retrieved runbook? (catches
  hallucination)
- **Answer relevance** — does it actually address the question?

The judge is cheap and offline — treat its scores as *indicative*, not ground
truth (the README's caveat on LLM-as-judge).

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
model.eval()

def generate(messages, max_new_tokens=160):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

SYSTEM_RAG = ('You are an SRE assistant. Answer using ONLY the runbook context below. '
              "If the answer is not in the context, say you don't know.")

def rag_answer(question, k=3):
    hits = retrieve(question, chunks, index, k=k)
    ctx = '\n\n'.join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    ans = generate([{'role': 'system', 'content': SYSTEM_RAG},
                    {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {question}'}])
    return ans, ctx

In [ ]:
def judge_yes(prompt):
    out = generate([{'role': 'system', 'content': 'You are a strict evaluator. Reply with only YES or NO.'},
                    {'role': 'user', 'content': prompt}], max_new_tokens=4)
    return 1 if out.strip().upper().startswith('Y') else 0

def faithfulness(answer, ctx):
    return judge_yes(f'CONTEXT:\n{ctx}\n\nANSWER:\n{answer}\n\n'
                     'Is every factual claim in the ANSWER directly supported by the CONTEXT? YES or NO.')

def relevance(answer, question):
    return judge_yes(f'QUESTION: {question}\n\nANSWER: {answer}\n\n'
                     'Does the ANSWER directly address the QUESTION? YES or NO.')

rows = []
for q in QA:
    ans, ctx = rag_answer(q['question'], k=3)
    rows.append({'question': q['question'][:42] + '...',
                 'faithful': faithfulness(ans, ctx),
                 'relevant': relevance(ans, q['question'])})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nfaithfulness: {df['faithful'].mean():.2f}   answer-relevance: {df['relevant'].mean():.2f}")

## 7 · A faithfulness *failure* on purpose

Feed the model the **wrong** runbook for a question and judge it. A faithful model
should refuse ("don't know") rather than answer from an unrelated runbook — and the
judge should catch any answer that strays beyond the context.

In [ ]:
q = 'What caused the payments outage on 12 April 2026?'
wrong_ctx = next(c['text'] for c in chunks if c['doc_id'] == 'k8s-probes')
ans = generate([{'role': 'system', 'content': SYSTEM_RAG},
                {'role': 'user', 'content': f'CONTEXT:\n{wrong_ctx}\n\nQUESTION: {q}'}])
print('ANSWER:', textwrap.fill(ans, 88))
print('faithful?', 'YES' if faithfulness(ans, wrong_ctx) else 'NO',
      '| relevant?', 'YES' if relevance(ans, q) else 'NO')

## Takeaways

- **Two stages, two metrics.** Retrieval (recall@k, MRR) and generation
  (faithfulness, relevance) fail independently — measure both, and diagnose by
  stage (README Phase 7).
- **top-k** trades recall for noise + tokens; a **reranker** lifts MRR by reading
  query and chunk together — the biggest precision lever after chunking.
- **Chunk size** visibly moves retrieval quality — the highest-leverage knob, and
  the cheapest to sweep.
- **Faithfulness** is the hallucination guard: a fluent answer unsupported by the
  runbook must score 0. LLM-as-judge makes this cheap, but it's indicative, not
  ground truth.
- Always report quality **with** latency and cost — never a single number.

That closes the Data layer: **Lab 1** built a grounded, cited SRE assistant;
**Lab 2** put numbers on it and showed which knobs to turn. Next layer up:
**[04-orchestration](../../../04-orchestration/)**, where retrieval becomes a tool
an agent calls in a loop.